# Notebook 05 — Customer Segmentation

## Objective

Membangun segmentasi customer berdasarkan perilaku pembelian menggunakan
RFM (Recency, Frequency, Monetary) dan algoritma K-Means.

## Business Context

Perusahaan memiliki 10.000 customer dengan karakteristik dan perilaku
pembelian yang berbeda.

Segmentasi bertujuan untuk menemukan kelompok customer yang memiliki
karakteristik perilaku serupa sehingga perusahaan dapat:

- menentukan prioritas customer,
- membuat strategi marketing yang lebih terarah,
- mengidentifikasi customer bernilai tinggi,
- menemukan customer yang berpotensi membutuhkan re-engagement.

## Methodology

Pipeline segmentasi:

1. Load customer feature dataset
2. Select RFM features
3. Handle customers without completed transactions
4. Inspect RFM distributions
5. Transform and standardize features
6. Test multiple K-Means configurations
7. Evaluate clusters using Silhouette Score
8. Select an appropriate number of clusters
9. Profile each cluster
10. Translate clusters into business segments

## RFM Features

- **Recency** — jumlah hari sejak pembelian terakhir.
- **Frequency** — jumlah completed transactions.
- **Monetary** — total revenue dari completed transactions.

## Important Consideration

Customer yang belum memiliki completed transaction tidak memiliki nilai
Recency yang valid. Customer tersebut tidak akan dipaksakan memiliki nilai
Recency buatan untuk proses clustering.

Mereka akan dipisahkan terlebih dahulu dari populasi RFM dan dapat dianalisis
sebagai kelompok "No Purchase" setelah clustering.

## 1. Load Feature Dataset

### Input

`data/processed/customer_features.csv`

Dataset ini merupakan hasil dari Notebook 04 dan berisi satu baris untuk
setiap customer.

### Output

DataFrame `customer_features` yang akan digunakan sebagai sumber data
segmentasi.

### Why

Kita menggunakan feature dataset yang sudah divalidasi daripada menghitung
ulang RFM dari raw transaction data.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [6]:
PROCESSED_DIR = Path("../data/processed")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

FEATURE_PATH = PROCESSED_DIR / "customer_features.csv"

customer_features = pd.read_csv(FEATURE_PATH)

print("Customer feature dataset loaded.")
print("Shape:", customer_features.shape)

Customer feature dataset loaded.
Shape: (10000, 21)


## 2. Prepare RFM Dataset

Untuk clustering, kita menggunakan tiga feature utama:

- `recency_days`
- `frequency`
- `monetary`

Customer tanpa completed transaction memiliki `recency_days = NaN`.

Karena K-Means membutuhkan feature numerik tanpa missing value, customer
tersebut tidak dimasukkan ke dalam proses RFM clustering.

Mereka tetap dipertahankan dalam `customer_features` dan akan diberi
informasi segmentasi khusus setelah proses clustering selesai.

In [7]:
rfm_features = customer_features[
    [
        "customer_id",
        "recency_days",
        "frequency",
        "monetary",
        "has_purchase",
    ]
].copy()

rfm_customers = rfm_features[
    rfm_features["has_purchase"] == 1
].copy()

no_purchase_customers = rfm_features[
    rfm_features["has_purchase"] == 0
].copy()

print("Total customers:", len(rfm_features))
print("Customers used for RFM clustering:", len(rfm_customers))
print("Customers without purchase:", len(no_purchase_customers))
print("\nMissing values in clustering population:")
print(rfm_customers[["recency_days", "frequency", "monetary"]].isna().sum())

Total customers: 10000
Customers used for RFM clustering: 9730
Customers without purchase: 270

Missing values in clustering population:
recency_days    0
frequency       0
monetary        0
dtype: int64
